# Reorganize your existing project — on `tidy_work`

**Keep your existing GitHub repository. Do not create a new repository or fork.** This launcher creates a working branch called `tidy_work` in your existing repository. `main` stays unchanged until your instructor reviews and merges your work.

Run the numbered steps in **Google Colab**. “Copy into Colab” means Colab's temporary computer, not your laptop. No laptop download is required. Files are permanent only after you save them to GitHub (or external storage for large data).

Your instructor supplies this launcher, the toolkit URL, and a mapping URL prepared for your project. A mapping places your existing work in the template; it does not invent analysis. If you have no mapping, ask your instructor before starting. Start with **Runtime → Run cell** for each step; do not use Run all because you must inspect the preview before applying it.

The launcher never opens or merges a pull request. You make the commits; your instructor opens the review PR.

## 1. Copy the toolkit into Colab

Enter the toolkit repository URL supplied by your instructor. This is the repository containing `migration/` and `template/`, not your project repository. The code is copied into this runtime. Installation may take a minute.

In [ ]:
from pathlib import Path
import json, re, shutil, subprocess, sys, tempfile

TOOLKIT_URL = "https://github.com/juanatga/dsb_cap_template" # @param {type:"string"}
TOOLKIT_REF = "main" # @param {type:"string"}
if not re.fullmatch(r"https://github.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?", TOOLKIT_URL) or "/OWNER/" in TOOLKIT_URL:
    raise ValueError("Enter the instructor's actual toolkit repository URL.")
SESSION = Path.cwd() / "capstone-migration-session"
SESSION.mkdir(exist_ok=True)
TOOLKIT = SESSION / "toolkit"
if not TOOLKIT.exists():
    subprocess.run(["git", "clone", "--branch", TOOLKIT_REF, "--", TOOLKIT_URL, str(TOOLKIT)], check=True)
else:
    print("Using the toolkit already copied into this session. Start a fresh runtime for a toolkit update.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(TOOLKIT / "migration/requirements-dev.txt")], check=True)
if not shutil.which("gh"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "gh"], check=True)
sys.path.insert(0, str(TOOLKIT / "migration"))
import workflow
from common import safe
from validate import validate
print("Toolkit ready in Colab:", TOOLKIT)

## 2. Sign in to GitHub

Follow the displayed browser link and enter the one-time device code. Sign in as **yourself**, with access to your existing project repository. This code is not a personal access token. Never paste a password or token into a notebook cell.

Credentials remain in this temporary runtime. Do not share a live authenticated runtime. The last step lets you sign out.

In [ ]:
if subprocess.run(["gh", "auth", "status"], capture_output=True).returncode:
    subprocess.run(["gh", "auth", "login", "--hostname", "github.com", "--git-protocol", "https", "--web"], check=True)
subprocess.run(["gh", "auth", "setup-git", "--hostname", "github.com"], check=True)
account = json.loads(workflow.run(["gh", "api", "user"]))
print("Signed in as:", account["login"])
print("GitHub noreply email option:", str(account["id"]) + "+" + account["login"] + "@users.noreply.github.com")

## 3. Select your existing project and prepare `tidy_work`

Enter **your existing repository**, not the toolkit. Enter your name and a verified GitHub email, or use the noreply email shown above.

This step copies your repository into Colab and creates `tidy_work` from `main`. If the branch exists, it resumes it without resetting it. Unsaved local changes or conflicting remote changes stop the step. A different project needs a fresh runtime.

If this is a resumed migrated branch, skip steps 4–6 and go to the editing/readiness steps.

In [ ]:
PROJECT_URL = "https://github.com/OWNER/EXISTING_PROJECT" # @param {type:"string"}
COMMIT_NAME = "" # @param {type:"string"}
COMMIT_EMAIL = "" # @param {type:"string"}
PROJECT_URL = workflow.github_url(PROJECT_URL)
project_name = PROJECT_URL.removeprefix("https://github.com/").removesuffix(".git")
access = json.loads(workflow.run(["gh", "api", "repos/" + project_name]))
if not access.get("permissions", {}).get("push"):
    raise ValueError("Your signed-in account needs write access to this repository.")
REPO = workflow.prepare(PROJECT_URL, SESSION / "project")
workflow.set_identity(REPO, COMMIT_NAME, COMMIT_EMAIL)
BRANCH_URL = "https://github.com/" + project_name + "/tree/tidy_work"
print("Repository:", PROJECT_URL, "\nBranch: tidy_work\nColab files:", REPO)
print("GitHub branch (visible after first push):", BRANCH_URL)
print("Already migrated:", (REPO / ".migration/state.json").exists())

## 4. Load your instructor's mapping

Paste the **raw mapping URL** supplied by your instructor, starting with `https://raw.githubusercontent.com/`. This copies a small JSON file directly into Colab, not onto your computer. The mapping must describe the current version of your source files. If it is stale, ask your instructor to update it.

The mapping can include code adaptations. Use only a mapping reviewed by your instructor; migration itself does not execute that analysis code.

In [ ]:
from urllib.request import urlopen
from urllib.parse import urlparse
MAPPING_URL = "" # @param {type:"string"}
u = urlparse(MAPPING_URL)
if u.scheme != "https" or u.hostname != "raw.githubusercontent.com" or u.username or u.password:
    raise ValueError("Use the instructor's public raw.githubusercontent.com mapping URL.")
with urlopen(MAPPING_URL, timeout=30) as response:
    payload = response.read(5_000_001)
if len(payload) > 5_000_000:
    raise ValueError("Mapping is unexpectedly large; ask your instructor.")
json.loads(payload)
MAPPING = SESSION / "mapping.json"
MAPPING.write_bytes(payload)
print("Mapping copied into Colab:", MAPPING)

## 5. Preview — no project files change

Read the additions, updates, removals, and instructor notes. Removed files remain in earlier Git history after you commit; they disappear from the working branch's current layout. Unclassified files stop migration rather than being silently deleted.

If the branch already has a migration record, do not migrate again. Continue editing and checking it.

In [ ]:
DRAFT = SESSION / "draft"
APPROVED_PREVIEW = workflow.preview(REPO, MAPPING, TOOLKIT / "template", DRAFT)
print(json.dumps(APPROVED_PREVIEW["file_preview"], indent=2))
print("\nDetailed source actions:")
print(json.dumps(APPROVED_PREVIEW["actions"], indent=2))
print("\nInstructor notes:", APPROVED_PREVIEW["notes"])
print("Full mapping, including cell moves and exact adaptations:")
print(MAPPING.read_text())

## 6. Apply the reviewed migration in Colab

Type `MIGRATE` only after inspecting the preview. A draft is built and structurally checked in a separate temporary folder, then applied to `tidy_work` in Colab. **Nothing is saved to GitHub yet.**

Warnings about writing prompts and unpinned versions are expected in a draft. Structural errors stop migration. This step does not run your analysis.

In [ ]:
if input("Type MIGRATE to apply the preview: ").strip() != "MIGRATE":
    raise ValueError("Migration not applied.")
REPORT = workflow.apply(REPO, MAPPING, TOOLKIT / "template", DRAFT, APPROVED_PREVIEW)
print("Migration report (Colab only):", REPORT)
print(REPORT.read_text())
print("Project files are currently only in Colab. Save the branch in step 7.")

## 7. Review and save the branch to GitHub

Review the file summary, then type `SAVE`. This commits as you and pushes **only `tidy_work`** to your existing repository. `main` is unchanged. You can return to this step to save later edits and imported figures/data.

If GitHub has newer work, saving stops instead of overwriting it. Preserve your local changes and ask your instructor to help reconcile them. After saving, open the branch link and verify your files before ending the session.

In [ ]:
print("Repository:", PROJECT_URL, "\nBranch: tidy_work")
print(workflow.git(REPO, "status", "--short"))
print(workflow.git(REPO, "diff", "--stat"))
message = input("Short description of your changes: ").strip()
if input("Type SAVE to commit and push these changes: ").strip() != "SAVE":
    raise ValueError("Not saved; files remain only in Colab.")
commit = workflow.save(REPO, message)
print("Saved commit:", commit)
print("Verify your files on GitHub:", BRANCH_URL)

## 8. Finish the notebooks in Colab

Open notebook 01, then notebook 02 using the links below. Each may run in a separate Colab runtime. Follow SETUP.md to clone the project branch into that runtime, install dependencies, and acquire the data. Opening the notebook alone does not copy the rest of your project.

After editing, use **File → Save a copy in GitHub** and select your existing repository, branch **`tidy_work`**, and the matching `notebooks/01_eda.ipynb` or `notebooks/02_modelling.ipynb` path. Verify the saved notebook on GitHub. Ordinary Save may save a Drive copy instead; check the destination.

Notebook 02 needs processed outputs from notebook 01. For small outputs, save them to the branch using step 9 before opening 02. For large outputs, use the external location and acquisition instructions recorded in data/README.md. A notebook save does not save generated data or figures.

In [ ]:
for name in ["01_eda.ipynb", "02_modelling.ipynb"]:
    print(name, "https://colab.research.google.com/github/" + project_name + "/blob/tidy_work/notebooks/" + name)
print("Setup instructions:", "https://github.com/" + project_name + "/blob/tidy_work/SETUP.md")

## 9. Optional: transfer generated data and figures without using your laptop

Separate Colab runtimes do not share files. Use a temporary Google Drive folder to transfer small outputs from the **analysis runtime** to this **launcher runtime**.

In the analysis notebook, run this in a temporary transfer cell (not as part of the final analysis):

```python
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
transfer = Path('/content/drive/MyDrive/capstone-transfer')
# Select only the actual output files you want to save; replace this example.
relative_files = ['docs/result.png', 'data/processed/cleaned.csv']
for relative in relative_files:
    source = ROOT / relative
    target = transfer / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source, target)
```

Clear old files from your transfer folder before each transfer. Files larger than 25 MB must stay in documented external storage, not in this transfer set.

Back in this launcher, run the following cell. It first loads the latest saved notebooks from `tidy_work`, then imports the files you selected. **Return to step 7 to save them to GitHub.** This does not automatically save them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
TRANSFER_FOLDER = "capstone-transfer" # @param {type:"string"}
transfer = safe(Path('/content/drive/MyDrive'), TRANSFER_FOLDER)
workflow.sync(REPO)
from common import inventory
print("Files to import:", list(inventory(transfer)))
if input("Type IMPORT to copy these outputs into tidy_work: ").strip() != "IMPORT":
    raise ValueError("Nothing imported.")
print("Imported:", workflow.import_artifacts(REPO, transfer))
print("Files are only in the launcher checkout. Return to step 7 to commit and push.")

## 10. Check the latest saved work and submit

Save all local changes first. This step brings in the latest `tidy_work` commits and checks the files. Structural checks inspect layout; readiness checks also find remaining prompts and unpinned dependencies. Neither executes your notebooks or judges scientific correctness.

Manual checks remain unverified until your instructor records evidence. Run both notebooks from fresh sessions, complete the report's student tasks, then send your instructor the branch link. The instructor opens the pull request; you do not merge it.

In [ ]:
workflow.sync(REPO)
for mode in ["structure", "readiness"]:
    findings = validate(REPO, mode=mode)
    print("\n", mode.upper())
    for f in findings:
        print(f["level"].upper(), f["path"], f["message"])
    print(sum(f["level"] == "error" for f in findings), "errors")
print("Send this branch to your instructor when student tasks are finished:", BRANCH_URL)

## 11. End the session

First verify your saved files on GitHub. Sign out below, then use Colab's runtime menu to disconnect and delete the runtime. Do not delete the runtime while it contains unsaved work. Your GitHub repository and temporary Drive transfer folder are separate from the runtime.

In [ ]:
if input("Verified your files on GitHub? Type SIGN OUT: ").strip() == "SIGN OUT":
    subprocess.run(["gh", "auth", "logout", "--hostname", "github.com", "--user", account["login"]], input="Y\n", text=True, check=True)
    print("Signed out. You can now disconnect and delete the runtime.")